In [10]:
from io import BytesIO
import openpyxl
from openpyxl.drawing.image import Image as OpenpyxlImage
from IPython.display import HTML, display
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import plotly.express as px
from scipy.stats.qmc import LatinHypercube

# =====================================================================
# 1. DESIGN SPACE GENERATION (32 UNIQUE FORMULAS)
# =====================================================================
n_unique = 32
sampler = LatinHypercube(d=3, seed=42)
lhs_sample = sampler.random(n=n_unique)

# Bounds: [HEMA (%), PEGDA (%), LAP (%)]
lower_bounds = [40.0, 18.0, 0.05]
upper_bounds = [52.0, 24.0, 0.50]

scaled = lower_bounds + lhs_sample * (
    np.array(upper_bounds) - np.array(lower_bounds)
)

recetas = []
for i in range(n_unique):
  hema = round(scaled[i][0], 2)
  pegda = round(scaled[i][1], 2)
  pi_val = round(scaled[i][2], 3)

  # Water balances formulation to 100%
  water = round(100 - (hema + pegda + pi_val), 3)

  recetas.append({
      "Formula_ID": f"Formula_{i+1}",
      "HEMA (%)": hema,
      "PEGDA (%)": pegda,
      "LAP (%)": pi_val,
      "Water (%)": water,
  })

df_unique = pd.DataFrame(recetas)

# =====================================================================
# 2. WORKLIST GENERATION FOR 1 PLATE (96 WELLS TOTAL: 32 x 3 REPLICATES)
# =====================================================================
sbs_wells = [f"{row}{col}" for row in "ABCDEFGH" for col in range(1, 13)]

samples = []
for rep in [1, 2, 3]:
  for r in recetas:
    rc = r.copy()
    rc["Plate"] = "Plate_1"
    rc["Replicate"] = f"Rep_{rep}"
    samples.append(rc)

df_plate = pd.DataFrame(samples)

# Randomize well distribution across 96 wells
df_plate_random = df_plate.sample(frac=1, random_state=42).reset_index(drop=True)
df_plate_random.insert(1, "Well", sbs_wells)

df_master = df_plate_random[[
    "Plate",
    "Well",
    "Formula_ID",
    "Replicate",
    "HEMA (%)",
    "PEGDA (%)",
    "LAP (%)",
    "Water (%)",
]]

# =====================================================================
# 3. EXCEL EXPORT (WITH EMBEDDED DASHBOARD)
# =====================================================================
file_path = "Opentrons_ASMI_LHS_Design_32Formulas.xlsx"

# Generate static Matplotlib 3D plot to embed in Excel
fig_static = plt.figure(figsize=(9, 6))
ax_static = fig_static.add_subplot(111, projection="3d")
sc_static = ax_static.scatter(
    df_unique["HEMA (%)"],
    df_unique["PEGDA (%)"],
    df_unique["LAP (%)"],
    c=df_unique["LAP (%)"],
    cmap="turbo",
    s=60,
    edgecolors="black",
)
ax_static.set_title(
    "LHS Design Space (32 Unique Formulations)", fontweight="bold"
)
ax_static.set_xlabel("HEMA (%)")
ax_static.set_ylabel("PEGDA (%)")
ax_static.set_zlabel("LAP (%)")

img_stream = BytesIO()
plt.savefig(img_stream, format="png", dpi=120)
plt.close(fig_static)

# Save data sheets
with pd.ExcelWriter(file_path, engine="openpyxl") as writer:
  df_master.to_excel(writer, sheet_name="Opentrons_Worklist", index=False)
  df_unique.to_excel(writer, sheet_name="Unique_Formulations", index=False)

# Embed Dashboard sheet with image
wb = openpyxl.load_workbook(file_path)
ws_vis = wb.create_sheet(title="Dashboard_Graph", index=0)
img_stream.seek(0)
ws_vis.add_image(OpenpyxlImage(img_stream), "B2")
wb.save(file_path)

# =====================================================================
# 4. NOTEBOOK DISPLAY (SCROLLABLE TABLE & INTERACTIVE 3D CUBE)
# =====================================================================
print(f"✅ Excel file successfully generated: {file_path}\n")

print("📌 Opentrons Master Worklist Preview:")
table_html = f"""
<div style='max-height: 250px; overflow-y: auto; overflow-x: auto; border: 2px solid #002D56; border-radius: 5px; padding: 10px; margin-bottom: 20px;'>
    {df_master.to_html(classes='table table-striped', index=False, justify='center')}
</div>
"""
display(HTML(table_html))

print("📌 Interactive 3D LHS Space Cube (32 Unique Formulas):")
fig_interactive = px.scatter_3d(
    df_unique,
    x="HEMA (%)",
    y="PEGDA (%)",
    z="LAP (%)",
    color="LAP (%)",
    hover_name="Formula_ID",
    hover_data={
        "HEMA (%)": ":.2f",
        "PEGDA (%)": ":.2f",
        "LAP (%)": ":.3f",
        "Water (%)": ":.3f",
    },
    color_continuous_scale="Turbo",
    title="LHS Design Space - 32 Unique Hydrogel Formulas",
)

fig_interactive.update_traces(
    marker=dict(
        size=8,
        line=dict(width=1, color="black"),  # Dark border
        opacity=0.9,
    ),
    hovertemplate="<b>%{hovertext}</b><br><br>"
    + "HEMA: %{x}%<br>"
    + "PEGDA: %{y}%<br>"
    + "LAP: %{z}%<br>"
    + "Water: %{customdata[3]}%<extra></extra>",
)

fig_interactive.update_layout(
    template="plotly_white",
    margin=dict(l=0, r=0, b=0, t=50),
    font=dict(family="Arial, sans-serif", size=12, color="#222222"),
    title=dict(
        text="<b>Interactive LHS Design Space (32 Unique In-Well Solutions)</b>",
        x=0.5,
        y=0.95,
        font=dict(size=16, color="#002D56"),
    ),
    scene=dict(
        xaxis=dict(
            title="<b>HEMA (%)</b>",
            backgroundcolor="rgb(232, 237, 247)",
            showbackground=True,
            gridcolor="white",
            gridwidth=2,
        ),
        yaxis=dict(
            title="<b>PEGDA (%)</b>",
            backgroundcolor="rgb(232, 237, 247)",
            showbackground=True,
            gridcolor="white",
            gridwidth=2,
        ),
        zaxis=dict(
            title="<b>LAP (%)</b>",
            backgroundcolor="rgb(232, 237, 247)",
            showbackground=True,
            gridcolor="white",
            gridwidth=2,
        ),
        camera=dict(eye=dict(x=1.4, y=1.4, z=1.1)),
    ),
    coloraxis_colorbar=dict(
        title="LAP (%)",
        thicknessmode="pixels",
        thickness=15,
        lenmode="pixels",
        len=220,
        yanchor="top",
        y=0.9,
        ticks="outside",
    ),
)

fig_interactive.show()

✅ Excel file successfully generated: Opentrons_ASMI_LHS_Design_32Formulas.xlsx

📌 Opentrons Master Worklist Preview:


Plate,Well,Formula_ID,Replicate,HEMA (%),PEGDA (%),LAP (%),Water (%)
Plate_1,A1,Formula_17,Rep_3,43.12,22.66,0.216,34.004
Plate_1,A2,Formula_14,Rep_3,47.25,23.17,0.080,29.500
Plate_1,A3,Formula_10,Rep_3,47.11,19.28,0.406,33.204
Plate_1,A4,Formula_31,Rep_3,46.69,18.43,0.339,34.541
Plate_1,A5,Formula_2,Rep_2,43.49,18.73,0.360,37.420
Plate_1,A6,Formula_16,Rep_3,47.95,21.86,0.327,29.863
Plate_1,A7,Formula_6,Rep_3,42.54,19.40,0.119,37.941
Plate_1,A8,Formula_11,Rep_2,40.47,18.94,0.186,40.404
Plate_1,A9,Formula_1,Rep_1,51.71,23.92,0.094,24.276
Plate_1,A10,Formula_11,Rep_1,40.47,18.94,0.186,40.404


📌 Interactive 3D LHS Space Cube (32 Unique Formulas):
